# sub045_4_fold_soup — media uniforme de pesos de swa_f0..f4 de exp045 + BN recalculada (ver SPEC.md)

In [ ]:
SUB = "sub045_4_fold_soup"
N_BN = 800
BN_BS = 4
GOLD_VETO = 0.9201083029612395 - 0.02
CORPUS_OVERRIDE = "/workspace/data/cache/raptor96_384"
N_WORKERS_OVERRIDE = 8


## Setup: celdas de exp045

In [ ]:
# Ejecuta las celdas 2/4/6/8/10 del notebook de exp045 (mismo código de datos, modelo, predict y score)
import json as _json, time as _time
from pathlib import Path as _P
_here = _P.cwd(); _repo = next(p for p in [_here, *_here.parents] if (p / "src" / "knee").exists())
_src = _json.load(open(_repo / "experiments/exp045_raptor_aug_espacial/exp045_raptor_aug_espacial.ipynb", encoding="utf-8"))
_cells = ["".join(c["source"]) for c in _src["cells"]]
exec(_cells[2])
SMOKE = False; FOLDS = [0]; CORPUS = CORPUS_OVERRIDE; N_WORKERS = N_WORKERS_OVERRIDE
for _i in (4, 6, 8, 10):
    exec(_cells[_i])
import torch.nn as nn
BASE = "/workspace/users/rodrigo"
SWA_PATH = {0: f"{BASE}/wt_exp045_raptor_aug_espacial/results/exp045_raptor_aug_espacial/swa_f0.pt",
            **{k: f"{BASE}/wt_exp045_f{k}/results/exp045_raptor_aug_espacial/swa_f{k}.pt" for k in (1, 2, 3, 4)},
            "alldata": f"{BASE}/wt_exp045_raptor_aug_espacial/results/exp045_raptor_aug_espacial/swa_alldata.pt"}
OUT = _repo / "results" / SUB; OUT.mkdir(parents=True, exist_ok=True)
NONGOLD = [u for u in soft.index if u not in set(gold_ids) and u in ID2ROW]

def load_model(state):
    bb = build_backbone(ARCH, pretrained=False)
    m = RaptorClassifier(bb, F_dim=bb.num_features).to(DEVICE)
    m.load_state_dict(state); return m

def bn_sample(ids, n=None):
    ids = sorted(ids); rs = np.random.RandomState(0)
    return [ids[i] for i in sorted(rs.choice(len(ids), min(n or N_BN, len(ids)), replace=False))]

@torch.no_grad()
def recalc_bn(model, ids):
    bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
    for m in bns:
        m.reset_running_stats(); m.momentum = None
    model.eval()
    for m in bns:
        m.train()
    t = _time.time()
    for x, g, idx, _sag in loader(ids, K_EVAL, False, BN_BS):
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            model.encode(to_input(x, g))
    model.eval()
    print(f"BN recalculada en {len(bns)} capas con {len(ids)} estudios ({(_time.time() - t) / 60:.1f} min)", flush=True)
    return model

def evaluate(model, va_ids, name):
    t = _time.time()
    pv, pg = predict(model, va_ids), predict(model, gold_ids)
    sv, sg = score(pv), score(pg)
    if len(pv): pv.add_prefix("pred_").reset_index().to_csv(OUT / f"oof_{name}.csv", index=False)
    pg.add_prefix("pred_").reset_index().to_csv(OUT / f"gold_{name}.csv", index=False)
    r = {"pseudo_auc": sv["pseudo"]["mean"] if sv["pseudo"] else None, "v1_auc": sv["v1"]["mean"] if sv["v1"] else None,
         "gold_auc": sg["gold"]["mean"], "pseudo_per_label": sv["pseudo"]["per_label"] if sv["pseudo"] else None,
         "gold_per_label": sg["gold"]["per_label"], "n_val": len(va_ids), "minutes": (_time.time() - t) / 60}
    print(f"[{name}] pseudo L2 {r['pseudo_auc'] or float('nan'):.4f} | v1 {r['v1_auc'] or float('nan'):.4f} | gold {r['gold_auc']:.4f} | {r['minutes']:.1f} min", flush=True)
    return r

def save_metrics(m):
    m.update({"schema_version": 1, "exp": SUB, "padre": "exp045_raptor_aug_espacial", "author": AUTHOR, "git": GIT,
              "written_utc": datetime.now(timezone.utc).isoformat()})
    (OUT / "metrics.json").write_text(_json.dumps(m, indent=1, default=float))
    print("metrics ->", OUT / "metrics.json", flush=True)


## Soup

In [ ]:
states = [torch.load(SWA_PATH[k], map_location="cpu", weights_only=False)["model"] for k in range(5)]
soup = average_states(states); del states
_, va0 = fold_ids(0)
M = {"nota": "pseudo f0 leaky (el soup vio el val de fold 0 via folds 1-4): solo informativa; gold solo veta; decide el LB", "n_bn": N_BN}
model = load_model(soup)
M["soup_sin_bn"] = evaluate(model, va0, "soup_nobn")
recalc_bn(model, bn_sample(NONGOLD))
M["soup_bn"] = evaluate(model, va0, "soup_bn")
M["veto"] = "killed" if M["soup_bn"]["gold_auc"] < GOLD_VETO else "pasa_veto (decide LB)"
torch.save({"model": model.state_dict(), "arch": ARCH, "res": RES, "lab": LABELS, "exp": SUB,
            "soup_of": [SWA_PATH[k] for k in range(5)], "bn_recalc": {"n": N_BN}}, OUT / "soup_exp045.pt")
M["soup_ckpt"] = str(OUT / "soup_exp045.pt")
save_metrics(M)
print("VETO", M["veto"], flush=True)
